# Week 2 Day 3 Exercise: Local LLM Integration with Ollama

I vibe coded from cloud APIs to local hosting using **Ollama**. implemented:
1. Multi-model orchestration using `glm-5.1`, `qwen3.5`, and `kimi-k2.5`.
2. Agentic handoffs between a Manager and specialized SDR agents.
3. Local tool calling and formatting.

In [1]:
import os
import asyncio
from typing import Dict
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel

# Custom agents library imports
from agents import (
    Agent, 
    Runner, 
    trace, 
    function_tool, 
    OpenAIChatCompletionsModel, 
    input_guardrail, 
    GuardrailFunctionOutput
)

load_dotenv(override=True)

True

### Configure Ollama Client
We use the OpenAI-compatible endpoint provided by Ollama (typically port 11434).

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama" # Dummy key for compatibility

ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)

# Define local models
glm_model = OpenAIChatCompletionsModel(model="minimax-m2.7:cloud", openai_client=ollama_client)
qwen_model = OpenAIChatCompletionsModel(model="qwen3.5:cloud", openai_client=ollama_client)
kimi_model = OpenAIChatCompletionsModel(model="kimi-k2.5:cloud", openai_client=ollama_client)

### Define Specialized SDR Agents

In [3]:
instructions1 = "You are a professional sales agent for ComplAI. You write serious cold emails for SOC2 compliance."
instructions2 = "You are a witty sales agent for ComplAI. You write engaging cold emails for SOC2 compliance."
instructions3 = "You are a concise sales agent for ComplAI. You write short, to-the-point emails."

sales_agent1 = Agent(name="MiniMax Sales Agent", instructions=instructions1, model=glm_model)
sales_agent2 = Agent(name="Qwen Sales Agent", instructions=instructions2, model=qwen_model)
sales_agent3 = Agent(name="Kimi Sales Agent", instructions=instructions3, model=kimi_model)

description = "Write a cold sales email"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

### Define Infrastructure Tools & Email Manager

In [4]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    import sendgrid
    from sendgrid.helpers.mail import Mail, Email, To, Content
    
    # 1. Initialize the real SendGrid client
    # Make sure 'SENDGRID_API_KEY' is in your .env file!
    api_key = os.environ.get('SENDGRID_API_KEY')
    if not api_key:
        return {"status": "error", "message": "SendGrid API Key missing"}
        
    sg = sendgrid.SendGridAPIClient(api_key=api_key)
    
    # 2. Configure the email
    from_email = Email("pgaitond@gmail.com") # Must be a SendGrid verified sender
    to_email = To("pgaitond@gmail.com")      # Change to your actual recipient
    content = Content("text/html", html_body)
    
    # 3. Create the mail object
    mail = Mail(from_email, to_email, subject, content).get()
    
    # 4. EXECUTE THE ACTUAL SEND
    try:
        response = sg.client.mail.send.post(request_body=mail)
        if response.status_code >= 200 and response.status_code < 300:
            return {"status": "success", "status_code": response.status_code}
        else:
            return {"status": "error", "message": f"Status code {response.status_code}"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

### Orchestration: The Sales Manager

In [5]:
# Tools for the Manager
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description="Write a cold sales email")
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description="Write a cold sales email")

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body. """
    print(f"--- SENDING EMAIL ---\nSubject: {subject}\nBody: {html_body[:100]}...")
    # Integration logic for SendGrid would go here
    return {"status": "success"}

# --- Formatting Agents (also using local models) ---
subject_writer = Agent(name="Subject Writer", instructions="Write catchy subjects.", model=qwen_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write email subject")

html_converter = Agent(name="HTML Converter", instructions="Convert text to HTML.", model=qwen_model)
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert body to HTML")

emailer_agent = Agent(
    name="Email Manager",
    instructions="Format with tools and then send the email.",
    tools=[subject_tool, html_tool, send_html_email],
    model=qwen_model
)

In [6]:
sales_manager_instructions = """
1. Use all three sales_agent tools to generate three different email drafts.
2. Select the single most effective draft.
3. Handoff for Sending: You MUST call the 'Email Manager' tool immediately after 
selecting the winning draft. Do not summarize the email to the user; 
your ONLY final action is to hand off to the Email Manager."""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=[tool1, tool2, tool3],
    handoffs=[emailer_agent],
    model=glm_model
)

### Execute Workflow
Ensure Ollama is running before executing this cell.

In [7]:
message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Ollama Local SDR Workflow"):
    result = await Runner.run(sales_manager, message)
    print("Workflow result summary:", result.final_output)

--- SENDING EMAIL ---
Subject: Cut Your SOC2 Compliance Timeline in Half
Body: <!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content...
Workflow result summary: ✅ **Email Successfully Sent!**

Here's a summary of what was delivered:

| Component | Details |
|-----------|---------|
| **Subject** | Cut Your SOC2 Compliance Timeline in Half |
| **Recipient** | Dear CEO |
| **Sender** | Head of Business Development, ComplAI |
| **Format** | Professional HTML email |
| **Status** | ✅ Sent successfully |

---

**Key Elements of the Email:**

- 🎯 **Hook:** Relatable pain point (9-month SOC2 timeline)
- 📊 **Social Proof:** 200+ startups helped
- 💡 **Value Prop:** Audit-ready in weeks, not months
- 📋 **Clear Benefits:** 3 bullet points on deliverables
- 🤝 **Call-to-Action:** 15-minute meeting request

---

The email is now in the CEO's inbox. Would you like me to help you with a follow-up sequence or create variations for A/B testing?
